In [1]:
import pandas as pd
import sys
print("pandas OK:", pd.__version__)
print("Python:", sys.executable)

pandas OK: 3.0.1
Python: A:\Project\.venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../Data/diabetes.csv')
print(f"Loaded: {df.shape}")

Loaded: (253680, 22)


In [3]:
model         = joblib.load('../models/best_model.pkl')
scaler        = joblib.load('../models/scaler.pkl')
feature_names = joblib.load('../models/feature_names.pkl')

X        = df[feature_names]
X_scaled = scaler.transform(X)

df['Predicted_Diabetes'] = model.predict(X_scaled)
df['Risk_Probability']   = (model.predict_proba(X_scaled)[:, 1] * 100).round(1)

df['Diabetes_Status']  = df['Diabetes_binary'].map({0:'No Diabetes', 1:'Diabetes'})
df['Predicted_Status'] = df['Predicted_Diabetes'].map({0:'No Diabetes', 1:'Diabetes'})
df['Risk_Level']       = pd.cut(df['Risk_Probability'],
                                bins=[0,30,60,100],
                                labels=['Low Risk','Moderate Risk','High Risk'])
df['Age_Group']        = df['Age'].map({
                            1:'18-24', 2:'25-29', 3:'30-34', 4:'35-39',
                            5:'40-44', 6:'45-49', 7:'50-54', 8:'55-59',
                            9:'60-64', 10:'65-69', 11:'70-74', 12:'75-79', 13:'80+'})
df['BMI_Category']     = pd.cut(df['BMI'],
                                bins=[0,18.5,25,30,100],
                                labels=['Underweight','Normal','Overweight','Obese'])
df['Sex_Label']        = df['Sex'].map({0:'Female', 1:'Male'})
df['GenHlth_Label']    = df['GenHlth'].map({
                            1:'Excellent', 2:'Very Good', 3:'Good',
                            4:'Fair', 5:'Poor'})

print("All columns added successfully!")
print(f"Total columns now: {df.shape[1]}")

All columns added successfully!
Total columns now: 31


In [4]:
import os

df.to_csv('../Data/diabetes_powerbi.csv', index=False)

summary = pd.DataFrame({
    'Metric': [
        'Total Patients', 'Diabetic Patients',
        'Non Diabetic Patients', 'Diabetes Rate %',
        'Avg BMI Diabetic', 'Avg BMI Non Diabetic',
        'Model ROC AUC', 'Model Recall %'
    ],
    'Value': [
        len(df),
        int(df['Diabetes_binary'].sum()),
        int((df['Diabetes_binary']==0).sum()),
        round(df['Diabetes_binary'].mean()*100, 1),
        round(df[df['Diabetes_binary']==1]['BMI'].mean(), 1),
        round(df[df['Diabetes_binary']==0]['BMI'].mean(), 1),
        0.822,
        76.5
    ]
})

summary.to_csv('../Data/model_summary.csv', index=False)

size = os.path.getsize('../Data/diabetes_powerbi.csv') / (1024*1024)
print(f"diabetes_powerbi.csv saved! Size: {size:.1f} MB")
print(f"model_summary.csv saved!")
print(f"\nYour Data folder now has:")
for f in os.listdir('../Data'):
    print(f"  {f}")

diabetes_powerbi.csv saved! Size: 38.3 MB
model_summary.csv saved!

Your Data folder now has:
  diabetes.csv
  diabetes_012_health_indicators_BRFSS2015.csv
  diabetes_binary_5050split_health_indicators_BRFSS2015.csv
  diabetes_powerbi.csv
  model_summary.csv
